# Explainability Demo

This notebook demonstrates the LIME-based explainability features for understanding model predictions.

In [ ]:
import sys
sys.path.insert(0, '../backend')

from utils.explainability import ExplainabilityEngine
from models.lstm_model import LSTMModel
from models.bert_model import BERTModel
import torch
import numpy as np

print("Explainability modules loaded successfully")

## 1. Initialize Models and Explainability Engine

In [ ]:
# Load LSTM model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = LSTMModel(
    vocab_size=10000,
    embedding_dim=300,
    hidden_dim=128,
    output_dim=2,
    n_layers=2,
    bidirectional=True,
    dropout=0.3
)
lstm_model.load_state_dict(torch.load('../models/lstm_model.pth', map_location=device))
lstm_model.to(device)
lstm_model.eval()

# Initialize explainability engine
explainer = ExplainabilityEngine(model=lstm_model, model_type='lstm')

print("Models and explainability engine initialized!")

## 2. Sample News Article

In [ ]:
# Example fake news
fake_news = """
Scientists discover that chocolate consumption can increase intelligence by 300 percent.
Researchers at a major university found that eating chocolate daily leads to unprecedented cognitive improvements.
This breakthrough changes everything we know about nutrition.
"""

# Example real news
real_news = """
The World Health Organization released new guidelines on health recommendations.
According to latest research, the guidelines were updated based on scientific evidence.
Healthcare professionals worldwide are reviewing the new standards.
"""

print("Sample texts prepared for explanation")

## 3. Generate Explanations

In [ ]:
# Explain fake news prediction
print("Explaining FAKE news prediction:")
print("="*60)
fake_explanation = explainer.explain(fake_news, num_features=10)
print(f"\nPrediction: {fake_explanation['prediction']}")
print(f"Confidence: {fake_explanation['confidence']:.2%}")
print(f"\nTop contributing words (FAKE):")
for word, weight in fake_explanation['word_importance'].items():
    print(f"  {word:20s}: {weight:.4f}")

print("\n" + "="*60)
print("Explaining REAL news prediction:")
print("="*60)
real_explanation = explainer.explain(real_news, num_features=10)
print(f"\nPrediction: {real_explanation['prediction']}")
print(f"Confidence: {real_explanation['confidence']:.2%}")
print(f"\nTop contributing words (REAL):")
for word, weight in real_explanation['word_importance'].items():
    print(f"  {word:20s}: {weight:.4f}")

## 4. Visualize Explanations

In [ ]:
import matplotlib.pyplot as plt

# Plot word importance for FAKE news
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fake_words = list(fake_explanation['word_importance'].keys())
fake_weights = list(fake_explanation['word_importance'].values())

axes[0].barh(fake_words, fake_weights, color='#ef4444')
axes[0].set_xlabel('Feature Importance')
axes[0].set_title(f'FAKE News Explanation\nPrediction: {fake_explanation["prediction"]} ({fake_explanation["confidence"]:.1%})')
axes[0].invert_yaxis()

# Plot word importance for REAL news
real_words = list(real_explanation['word_importance'].keys())
real_weights = list(real_explanation['word_importance'].values())

axes[1].barh(real_words, real_weights, color='#22c55e')
axes[1].set_xlabel('Feature Importance')
axes[1].set_title(f'REAL News Explanation\nPrediction: {real_explanation["prediction"]} ({real_explanation["confidence"]:.1%})')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. SHAP Value Analysis

In [ ]:
# Alternative: SHAP-based explanation
# Note: SHAP requires additional setup, this is a demonstration

# Get SHAP values for the model
# shap_values = explainer.get_shap_explanation(fake_news)
# print(f"SHAP values shape: {shap_values.shape}")
# print(f"SHAP analysis would provide probability-based feature contributions")

print("LIME-based explanations are sufficient for model interpretability.")
print("SHAP analysis can be added for deeper probability-based insights.")

## 6. Comparison Analysis

In [ ]:
# Analyze why different predictions happen
mixed_news = """
Scientists at the university conducted research on nutrition and health benefits.
The study found interesting chocolate results that need verification.
Experts say more research is needed before drawing conclusions.
"""

mixed_explanation = explainer.explain(mixed_news, num_features=15)

print(f"Mixed News Analysis:")
print(f"Prediction: {mixed_explanation['prediction']}")
print(f"Confidence: {mixed_explanation['confidence']:.2%}")
print(f"\nThis shows how the model balances conflicting signals in the text.")
print(f"\nKey words contributing to prediction:")
for i, (word, weight) in enumerate(list(mixed_explanation['word_importance'].items())[:10], 1):
    direction = "supports FAKE" if weight > 0 else "supports REAL"
    print(f"  {i:2d}. {word:20s}: {weight:+.4f} ({direction})")

## Summary

- **LIME Explanations**: Provide interpretable word-level importance for predictions
- **Word Contribution**: Shows which words push the model toward FAKE/REAL classification
- **Confidence Calibration**: Higher confidence indicates stronger agreement from multiple features
- **Mixed Signals**: The explainability reveals how the model handles conflicting information
- **User Interface**: These explanations are displayed to users in the frontend for transparency